# Bone Cancer Detector — Colab GPU Training

Steps:
1. **Runtime > Change runtime type > T4 GPU** (then Runtime > Run all)
2. When asked, paste your Kaggle API token (`KGAT_...` from kaggle.com/settings/api)
3. At the end, `best_model.pt` downloads automatically — put it in your local `checkpoints/` folder

Trains EfficientNet-B3 (~12M params, 300px, 25 epochs) on 5,779 X-rays. Takes ~1 hour.

In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi -L

In [ ]:
!pip install -q kaggle scikit-learn tqdm
import os
from getpass import getpass
os.environ['KAGGLE_API_TOKEN'] = getpass('Paste Kaggle API token (KGAT_...): ')
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
print('Kaggle auth OK')
api.dataset_download_files('musakhanalabazy/bone-tumor-dataset3', path='/content/raw', unzip=True)
!ls /content/raw/dataset && ls /content/raw/dataset/train

In [ ]:
import shutil
from pathlib import Path

SRC = Path('/content/raw/dataset')
DST = Path('/content/data')
CLASSES = {0: 'benign', 1: 'malignant'}


def label_of(img: Path, lbl_dir: Path) -> str:
    label = lbl_dir / (img.stem + '.txt')
    if not label.exists():
        label = lbl_dir / (img.name + '.txt')
        if not label.exists():
            cands = list(lbl_dir.glob(img.stem + '*'))
            label = cands[0] if cands else None
    if label is None or not label.exists() or label.stat().st_size == 0:
        return 'normal'
    try:
        return CLASSES.get(int(float(open(label).read().strip().split()[0])), 'malignant')
    except Exception:
        return 'normal'


for split in ['train', 'val', 'test']:
    for c in ['normal', 'benign', 'malignant']:
        (DST / split / c).mkdir(parents=True, exist_ok=True)
    for img in (SRC / split / 'images').iterdir():
        if img.suffix.lower() in ('.jpg', '.jpeg', '.png'):
            shutil.copy2(img, DST / split / label_of(img, SRC / split / 'labels') / img.name)
for s in ['train', 'val', 'test']:
    print(s, {d.name: len(list(d.glob('*'))) for d in (DST / s).iterdir()})

In [ ]:
import json, time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

OUT = '/content/best_model.pt'
EPOCHS, BATCH, LR, IMG = 25, 48, 3e-4, 300
device = torch.device('cuda')

train_tf = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
train_ds = datasets.ImageFolder('/content/data/train', transform=train_tf)
val_ds = datasets.ImageFolder('/content/data/val', transform=eval_tf)
names = train_ds.classes
train_ld = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
val_ld = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

counts = torch.zeros(len(names))
for _, y in DataLoader(train_ds, batch_size=512, num_workers=2):
    for c in y.tolist():
        counts[c] += 1
weights = (counts.sum() / (len(names) * counts)).to(device)

model = efficientnet_b3(weights=EfficientNet_B3_Weights.IMAGENET1K_V1)
model.classifier = nn.Sequential(nn.Dropout(0.4), nn.Linear(model.classifier[1].in_features, len(names)))
model = model.to(device)
print('params(M):', round(sum(p.numel() for p in model.parameters()) / 1e6, 1))

crit = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = GradScaler()
best = 0.0

for ep in range(1, EPOCHS + 1):
    model.train()
    t0, ls, co, to = time.time(), 0.0, 0, 0
    for x, y in tqdm(train_ld, desc=f'Ep {ep}/{EPOCHS}', leave=False):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad()
        with autocast():
            logits = model(x)
            loss = crit(logits, y)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        ls += loss.item() * x.size(0)
        co += (logits.argmax(1) == y).sum().item()
        to += x.size(0)
    sched.step()
    model.eval()
    P, T = [], []
    with torch.no_grad():
        for x, y in val_ld:
            P.extend(model(x.to(device)).argmax(1).cpu().tolist())
            T.extend(y.tolist())
    va = accuracy_score(T, P)
    print(f'Ep{ep}: train={co/to:.4f} val={va:.4f} F1={f1_score(T, P, average="macro"):.4f} ({time.time()-t0:.0f}s)')
    if va > best:
        best = va
        torch.save({'model': model.state_dict(), 'classes': names, 'acc': va, 'arch': 'efficientnet_b3', 'img': IMG}, OUT)
        print(f'  *** saved ({va:.4f})')
print(f'DONE best_val={best:.4f}')

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, classification_report

ckpt = torch.load('/content/best_model.pt', map_location='cpu')
print('val:', round(float(ckpt['acc']), 4), 'arch:', ckpt.get('arch'))

from google.colab import files
files.download('/content/best_model.pt')
print('Downloaded. Copy it to your local checkpoints/best_model.pt and run eval_test.py.')